# Model Explainability (SHAP)
This notebook mirrors `src/explainability.py`. It computes SHAP values for the LightGBM claim frequency model to identify key risk drivers.

In [ ]:
import os
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import shap

from src.model_frequency import load_and_prep_data

# Load model
model_path = '../reports/lgbm_frequency_model.pkl'
with open(model_path, "rb") as f:
    pipeline = pickle.load(f)

preprocessor = pipeline.named_steps["preprocessor"]
lgb_model = pipeline.named_steps["model"]

# Load and sample data
_, X_test, _, _, _, _ = load_and_prep_data('../data/processed/features.csv')
X_sample = X_test.sample(n=5000, random_state=42)
X_sample_prep = preprocessor.transform(X_sample)

# Feature names
cat_names = preprocessor.named_transformers_["cat"].get_feature_names_out()
num_names = preprocessor.transformers_[0][2]
feature_names = list(num_names) + list(cat_names)


## Computing SHAP Values

In [ ]:
explainer = shap.TreeExplainer(lgb_model)
shap_values = explainer.shap_values(X_sample_prep)


## Feature Importance (Mean Absolute Impact)

In [ ]:
shap.summary_plot(shap_values, X_sample_prep, feature_names=feature_names, plot_type="bar", max_display=10)

## SHAP Beeswarm Plot (Directional Impact)

In [ ]:
shap.summary_plot(shap_values, X_sample_prep, feature_names=feature_names, max_display=10)